# 7장 임베딩과 의미 검색 (풀이)

검증용 풀이 노트북입니다. 공유하지 않습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(공개 LLM, 임베딩 모델, 리랭커)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |
| `EMBED_MODEL` | 설정하지 않아도 됩니다. 다른 임베딩 모델을 쓰거나 모델 파일이 든 폴더를 따로 받았을 때, 모델 이름이나 폴더 경로를 넣습니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install sentence-transformers rank-bm25 pypdf
# 7장에서 씁니다. 문서를 읽고 검색하는 함수는 같은 폴더의 corpus.py에 있습니다
import numpy as np

import corpus
import embed

catalog = corpus.catalog()               # {문서 id: 메타데이터}. 문서 120건
questions = corpus.evalset()             # 평가 질문 62개. 키의 뜻은 data/README.md에 있습니다
embed.load()                             # 임베딩 모델을 불러옵니다. 처음 한 번 약 470MB를 내려받습니다
print(f'문서 {len(catalog)}건, 평가 질문 {len(questions)}개')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 7-1. 낱말이 겹치지 않아도 가까운 문장

교재의 절: "코사인 유사도"

질문 하나와 문장 다섯 개의 코사인 유사도를 계산하고, 질문과 겹치는 키워드 토큰 수와 나란히 출력합니다. 정답 문장(캐필러리 교체 기준)은 질문과 낱말이 거의 겹치지 않고, 낱말이 많이 겹치는 문장(세라믹 머그컵)은 관계가 없습니다. 어느 문장이 1위인지, 관계없는 문장의 유사도가 얼마인지 봅니다.

In [ ]:
question = '와이어를 꿰어 볼을 누르는 세라믹 툴은 몇 번 쓰고 바꾸나요?'
sentences = ['캐필러리는 금선 기준 본딩 100만 회마다 교체한다.',
             '세라믹 머그컵은 몇 번 쓰고 바꾸나요?',
             '다이싱 블레이드는 누적 절단 거리가 10km에 이르면 교체한다.',
             '와이어 본더의 히터 블록은 리드 프레임을 200℃로 가열한다.',
             '클린룸 에어 샤워는 20초 동안 한다.']
question_vector = embed.query(question)              # 질문 앞에 'query: '를 붙여 벡터로 바꿉니다
sentence_vectors = embed.passages(sentences)         # 문장 앞에 'passage: '를 붙입니다. (5 × 384) 배열
similarity = sentence_vectors @ question_vector      # 길이가 1인 벡터끼리의 내적이 코사인 유사도입니다
shared = [len(set(corpus.words(question)) & set(corpus.words(sentence))) for sentence in sentences]
print(f'차원 {len(question_vector)}, 벡터의 길이 {np.linalg.norm(question_vector):.3f}')
for index in np.argsort(-similarity):
    print(f'유사도 {similarity[index]:.3f}, 겹치는 토큰 {shared[index]:>2}개: {sentences[index]}')

## 예제 7-2. 질문 유형마다 두 검색 비교하기

교재의 절: "두 검색이 잘 찾는 질문"

구판 5건을 뺀 유효 문서 115건의 본문 앞에 문서 번호와 제목을 붙여, 문서 하나에 벡터 하나씩 임베딩해 인덱스를 만듭니다(512토큰을 넘는 문서는 뒷부분이 잘립니다). 그리고 같은 문서로 키워드 검색(BM25) 인덱스도 만듭니다. "바꿔 말함" 질문 8개와 "코드" 질문 8개를 두 검색으로 찾아, 정답 문서가 1위에 온 질문 수와 1위를 놓친 질문의 순위를 봅니다. 임베딩에 1분쯤 걸립니다.

In [ ]:
valid = [doc_id for doc_id, info in catalog.items() if info['status'] == '유효']
texts = [f"{catalog[doc_id]['doc_no']} {catalog[doc_id]['title']}\n{corpus.read_text(doc_id)}" for doc_id in valid]
vector_index, keyword_index = corpus.VectorIndex(texts), corpus.KeywordIndex(texts)


def rank_of(scores, gold):
    """점수 순으로 줄 세웠을 때 정답 문서가 처음 나오는 순위"""
    order = [valid[index] for index in np.argsort(-scores)]
    return next(rank for rank, doc_id in enumerate(order, 1) if doc_id in gold)


for kind in ['바꿔 말함', '코드']:
    picked = [item for item in questions if item['type'] == kind]
    ranks = [(item['id'], rank_of(vector_index.scores(item['question']), item['docs']),
              rank_of(keyword_index.scores(item['question']), item['docs'])) for item in picked]
    print(f"{kind} {len(picked)}개: 정답 문서가 1위인 질문은 임베딩 {sum(row[1] == 1 for row in ranks)}개, "
          f"키워드 {sum(row[2] == 1 for row in ranks)}개")
    print('  1위를 놓친 질문(id, 임베딩 순위, 키워드 순위):', [row for row in ranks if max(row[1:]) > 1])

## 실습 7-1. 표현을 바꿔 묻기

교재의 절: "개념 확인 실습"

평가 질문 E12("와이어를 꿰어 볼을 누르는 세라믹 툴은 금선 기준으로 몇 번 쓰고 바꾸나요?")의 정답 문서는 캐필러리 교체 기준이 적힌 SOP-WB-002입니다. 같은 뜻의 질문을 표현만 바꿔 세 가지 이상 `variants`에 적습니다. 문서의 낱말("캐필러리", "교체")을 그대로 쓴 질문, 현장에서 부를 법한 다른 이름을 쓴 질문, 영어를 섞은 질문을 넣어 봅니다. 질문마다 두 검색 가운데 어느 쪽이 SOP-WB-002를 1위로 찾을지 `my_guess`에 먼저 적고 실행합니다. 예제 7-2의 인덱스를 쓰므로 예제 7-2를 먼저 실행합니다. 예측과 다른 결과가 나온 질문을 키워드 검색과 임베딩의 원리로 설명하면 끝납니다.

In [ ]:
variants = ['캐필러리는 몇 회 쓰고 교체하나요?',
            '본딩 툴 바꾸는 주기가 어떻게 되나요?',
            'capillary 교체 기준이 뭔가요?',
            '와이어 본더 바늘은 몇 번 찍으면 갈아요?']
my_guess = {variants[0]: '둘 다', variants[1]: '임베딩', variants[2]: '임베딩', variants[3]: '임베딩'}

for variant in variants:
    by_vector = rank_of(vector_index.scores(variant), ['SOP-WB-002'])
    by_keyword = rank_of(keyword_index.scores(variant), ['SOP-WB-002'])
    print(f'임베딩 {by_vector}위, 키워드 {by_keyword}위, 예측 {my_guess.get(variant)}: {variant}')